# SECCIÓN 1: MARCO CONCEPTUAL Y OBJETIVOS DE APRENDIZAJE

## Resultados de Aprendizaje Esperados (RAE)
Al finalizar esta sesión interactiva, el analista de datos será capaz de:
1. **Interpretar** diagramas de dispersión para identificar relaciones visuales entre variables continuas.
2. **Cuantificar** el grado de asociación lineal utilizando el Coeficiente de Correlación de Pearson ($r$).
3. **Modelar** relaciones predictivas bivariadas mediante Regresión Lineal Simple (Mínimos Cuadrados Ordinarios).
4. **Evaluar** la calidad del ajuste del modelo y analizar la distribución de los residuos para validar supuestos subyacentes.

## Hoja de Ruta Teórica
En las semanas previas, el análisis se ha centrado exclusivamente en el enfoque univariado: comprender la distribución, dispersión y tendencia central de vectores aislados. Sin embargo, los fenómenos reales rara vez operan en el vacío. El análisis bivariado constituye el andamiaje fundamental para entender la asociación entre dos variables métricas. En este laboratorio, realizaremos la transición formal desde la estadística descriptiva hacia la inferencia y el modelado predictivo, estableciendo las bases para modelos de *Machine Learning* más complejos.


# SECCIÓN 2: CARGA DE LIBRERÍAS Y CONFIGURACIÓN DEL ENTORNO

Para garantizar la reproducibilidad y el rigor en el tratamiento de los datos, importaremos el conjunto de herramientas estándar de la industria. Se ha configurado un tema sobrio y profesional para todas las salidas gráficas.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn import metrics

# Configuración del entorno gráfico (Estilo Springer/Académico)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.labelcolor'] = '#003865'
plt.rcParams['axes.titlecolor'] = '#003865'


# SECCIÓN 3: GENERACIÓN Y COMPRENSIÓN DEL DATASET DE TRABAJO

Para ilustrar los conceptos de dependencia lineal, generaremos un proceso estocástico que simula el comportamiento de 200 observaciones. La variable dependiente $Y$ (Calificación Final) está regida por la siguiente ecuación paramétrica con un componente de error gaussiano:

$$Y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \epsilon \quad \text{donde } \epsilon \sim \mathcal{N}(0, \sigma^2)$$

Donde $X_1$ representa las horas de estudio semanal y $X_2$ las horas de sueño diario.


In [ ]:
# Fijar la semilla para asegurar reproducibilidad
np.random.seed(42)

# Definición de parámetros poblacionales (N = 200)
N = 200
horas_estudio = np.random.normal(loc=15.0, scale=5.0, size=N)
horas_sueno = np.random.normal(loc=7.0, scale=1.5, size=N)

# Componente de error aleatorio (Ruido Blanco)
epsilon = np.random.normal(loc=0.0, scale=8.0, size=N)

# Ecuación estructural subyacente
# Y = 30 + 3.5(X1) + 2.0(X2) + error
calificacion_final = 30.0 + (3.5 * horas_estudio) + (2.0 * horas_sueno) + epsilon
calificacion_final = np.clip(calificacion_final, 0, 100) # Acotar lógicamente entre 0 y 100

# Consolidación en estructura tabular (DataFrame)
df = pd.DataFrame({
    'Horas_Estudio': horas_estudio,
    'Horas_Sueno': horas_sueno,
    'Calificacion': calificacion_final
})

# Exploración inicial
print("--- Información de la Estructura de Datos ---")
df.info()

print("\n--- Estadísticas Descriptivas ---")
display(df.describe())


# SECCIÓN 4: EXPLORACIÓN BIVARIADA VISUAL (GRÁFICOS DE DISPERSIÓN)

El Cuarteto de Anscombe demostró irrevocablemente que conjuntos de datos con propiedades estadísticas idénticas (media, varianza, correlación) pueden presentar distribuciones gráficas radicalmente distintas. Por consiguiente, el análisis exploratorio visual (EDA) es un requisito sine qua non previo a la cuantificación matemática.


In [ ]:
# Construcción del Gráfico de Dispersión
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df, 
    x='Horas_Estudio', 
    y='Calificacion', 
    color='#003865', 
    alpha=0.7, 
    edgecolor='w', 
    s=70
)

plt.title('Relación Bivariada: Horas de Estudio vs. Calificación Académica', weight='bold')
plt.xlabel('Horas de Estudio Semanales (X)')
plt.ylabel('Calificación Final (Y)')
plt.show()


**Pregunta de Reflexión:** Observe la nube de puntos generada. ¿La densidad de la varianza condicional de $Y$ dado $X$ parece constante a lo largo del dominio, o la dispersión se amplifica en ciertos rangos?


# SECCIÓN 5: CUANTIFICACIÓN MATEMÁTICA: COVARIANZA Y CORRELACIÓN DE PEARSON

La covarianza mide la variabilidad conjunta de dos variables aleatorias. No obstante, al depender de las escalas absolutas de medición de $X$ e $Y$, carece de interpretabilidad directa. Para subsanar este defecto intrínseco, se estandariza el estadístico mediante el producto de sus desviaciones estándar poblacionales/muestrales, obteniendo el **Coeficiente de Correlación de Pearson ($r_{XY}$)**.

$$r_{XY} = \frac{\sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_{i=1}^{n} (x_i - \bar{x})^2} \sqrt{\sum_{i=1}^{n} (y_i - \bar{y})^2}}$$


In [ ]:
# Cálculo de la matriz de correlación lineal de Pearson
matriz_correlacion = df.corr(method='pearson')

# Visualización mediante Mapa de Calor (Heatmap)
plt.figure(figsize=(7, 5))
sns.heatmap(
    matriz_correlacion, 
    annot=True, 
    fmt=".3f", 
    cmap='coolwarm', 
    vmin=-1, 
    vmax=1, 
    linewidths=0.5,
    cbar_kws={'label': 'Coeficiente de Correlación (r)'}
)
plt.title('Matriz de Correlación de Pearson', weight='bold')
plt.show()


**Cuestionario de Autoevaluación:**
* Si $r = +0.85$, la relación lineal es fuertemente proporcional y directa.
* Si $r = 0.00$, indica ausencia estricta de correlación *lineal* (aunque podría existir correlación no lineal).
* Si $r = -0.72$, la relación lineal es inversa y moderadamente fuerte.


# SECCIÓN 6: MODELADO PREDICTIVO — REGRESIÓN LINEAL SIMPLE (OLS)

Procedemos a estimar la ecuación teórica poblacional mediante el ajuste de una recta muestral:

$$\hat{Y} = \beta_0 + \beta_1 X$$

El principio algorítmico utilizado es el de **Mínimos Cuadrados Ordinarios (OLS - Ordinary Least Squares)**, cuyo objetivo analítico consiste en minimizar la Suma de los Errores Cuadráticos ($SSE$):

$$\text{Minimizar } SSE = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$


In [ ]:
# 1. Separación de Variables (Características y Objetivo)
X_feature = df[['Horas_Estudio']]  # Matriz de diseño (2D requerida por sklearn)
y_target = df['Calificacion']      # Vector objetivo (1D)

# 2. Instanciación y Ajuste del Estimador Lineal
modelo_ols = LinearRegression()
modelo_ols.fit(X_feature, y_target)

# 3. Extracción de Parámetros Estructurales
beta_0 = modelo_ols.intercept_
beta_1 = modelo_ols.coef_[0]

print("--- Parámetros del Modelo OLS Ajustado ---")
print(f"Intercepto (Beta_0): {beta_0:.4f}")
print(f"Pendiente (Beta_1):  {beta_1:.4f}")
print(f"Ecuación Predictiva: Calificación = {beta_0:.4f} + {beta_1:.4f} * Horas_Estudio")

# 4. Visualización de la Recta de Regresión
plt.figure(figsize=(9, 6))
sns.scatterplot(x=df['Horas_Estudio'], y=df['Calificacion'], color='#003865', alpha=0.6, label='Observaciones Empíricas')

# Generación del hiperplano (recta) ajustado
x_rango = np.linspace(df['Horas_Estudio'].min(), df['Horas_Estudio'].max(), 100)
y_predicha = beta_0 + beta_1 * x_rango
plt.plot(x_rango, y_predicha, color='#E85C0B', linewidth=2.5, label='Recta de Regresión OLS')

plt.title('Ajuste del Modelo de Regresión Lineal Simple', weight='bold')
plt.xlabel('Horas de Estudio Semanales (X)')
plt.ylabel('Calificación Final (Y)')
plt.legend()
plt.show()


# SECCIÓN 7: EVALUACIÓN DEL MODELO Y ANÁLISIS DE RESIDUOS

Un modelo ajustado carece de validez sin un análisis estricto de sus métricas de error y el cumplimiento de sus supuestos.
1. **Coeficiente de Determinación ($R^2$):** Proporción de la varianza en $Y$ predecible a partir de $X$.
2. **Error Cuadrático Medio ($RMSE$):** Desviación estándar de los residuos no explicados (medido en las mismas unidades que $Y$).


In [ ]:
# Generación de predicciones sobre el conjunto de entrenamiento
y_estimada = modelo_ols.predict(X_feature)

# Cálculo de métricas de desempeño
r_cuadrado = metrics.r2_score(y_target, y_estimada)
rmse = np.sqrt(metrics.mean_squared_error(y_target, y_estimada))

print("--- Métricas de Desempeño del Modelo ---")
print(f"Coeficiente de Determinación (R^2): {r_cuadrado:.4f}")
print(f"Error Cuadrático Medio (RMSE):      {rmse:.4f}")

# Análisis de Residuos (e_i = Y - Y_hat)
residuos = y_target - y_estimada

plt.figure(figsize=(9, 5))
plt.scatter(y_estimada, residuos, color='#003865', alpha=0.6, edgecolor='w')
plt.axhline(y=0, color='#E85C0B', linestyle='--', linewidth=2)
plt.title('Análisis de Residuos: Comprobación de Homocedasticidad', weight='bold')
plt.xlabel('Valores Predichos ($\hat{Y}$)')
plt.ylabel('Residuos ($e_i$)')
plt.show()


**Análisis Diagnóstico:** Si la distribución de residuos conforma una franja horizontal uniforme alrededor del cero (como se espera observar), no existe evidencia sustancial para rechazar el supuesto de varianza constante (homocedasticidad).


# SECCIÓN 8: DISCUSIÓN CRÍTICA — CORRELACIÓN VS. CAUSALIDAD Y LIMITACIONES

Resulta fundamental distinguir la diferencia epistémica entre correlación observacional y causalidad mecánica. La presencia de un estadístico $r$ elevado, o un modelo predictivo con $R^2$ alto, no infiere que $X$ modifique a $Y$. Este sesgo usualmente obedece a las denominadas **Variables de Confusión (Lurking Variables)**.

Adicionalmente, el estimador de Pearson adolece de una seria limitación matemática: es enteramente ciego a dependencias funcionales de naturaleza no lineal.


# SECCIÓN 9: TALLER PRÁCTICO Y RETOS GRADUADOS (EVALUACIÓN FORMAL)

## Reto 1 (Nivel Aplicar - Guiado)
Calcule matemáticamente el coeficiente de correlación entre la variable `Horas_Sueno` y `Calificacion`. Construya el respectivo gráfico de dispersión.

## Reto 2 (Nivel Analizar - Intermedio)
Basándose en el modelo `modelo_ols` previamente ajustado, determine la calificación proyectada para un individuo que reporta una carga de 22 horas de estudio semanales. Ejecute la predicción tanto empleando la API de Scikit-Learn (`.predict()`) como mediante la resolución manual de la ecuación algebraica estimada.

## Reto 3 (Nivel Evaluar - Avanzado)
Implemente una simulación que genere un vector $X$ en el intervalo $[-10, 10]$ y un vector $Y$ tal que $Y = X^2 + \epsilon$. Proceda a calcular el coeficiente de correlación de Pearson entre ambas variables. Disponga una celda en formato Markdown argumentando analíticamente la divergencia entre el resultado estadístico ($r \approx 0$) y la dependencia estructural determinista de las variables.


In [ ]:
# --- Espacio de Resolución para el Reto 1 ---





In [ ]:
# --- Espacio de Resolución para el Reto 2 ---





In [ ]:
# --- Espacio de Resolución para el Reto 3 ---



